In [0]:
#this uses dbutils.fs.ls to inspect your storage folders before any processing starts.
print("--- checking full_load ---")
try:
    result = dbutils.fs.ls("/Volumes/workspace/tmdb/raw/full_load/")
    print(f"Found {len(result)} items:")
    for f in result:
        print(" ", f.path, "|", f.name, "|", f.size, "bytes")
except Exception as e:
    print("ERROR:", e)

print("\n--- listing the volume root ---")
try:
    for f in dbutils.fs.ls("/Volumes/workspace/tmdb/raw/"):
        print(" ", f.path, "(dir)" if f.isDir() else "(file)")
except Exception as e:
    print("ERROR:", e)

In [0]:
import uuid
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, LongType, DoubleType, BooleanType,
    ArrayType, TimestampType
)
from delta.tables import DeltaTable

# Widgets for parameterization (backfill-friendly)
dbutils.widgets.text("load_type", "full", "Load Type (full/incremental)")
dbutils.widgets.text("batch_id", datetime.now().strftime("%Y%m%d_%H%M%S"), "Batch ID")

load_type = dbutils.widgets.get("load_type")
batch_id  = dbutils.widgets.get("batch_id")

print(f"Executing pipeline with Load Type: '{load_type}', Batch ID: '{batch_id}'")

In [0]:
# Bronze is a RAW LANDING ZONE. We only define the ENVELOPE shape,
# and keep each nested record as a raw JSON string. Silver does the typing.

# ---- Full load envelope ----
full_load_envelope_schema = StructType([
    StructField("pulled_at",       StringType(), True),
    StructField("source_endpoint", StringType(), True),
    StructField("year_range",      StringType(), True),
    StructField("enriched",        BooleanType(), True),
    StructField("enrich_limit",    LongType(), True),
    StructField("record_count",    LongType(), True),
    StructField("results",         ArrayType(StringType()), True),  # <-- raw JSON strings
])

# ---- Incremental envelope ----
incremental_envelope_schema = StructType([
    StructField("pulled_at",      StringType(), True),
    StructField("trending_today", StringType(), True),   # <-- raw JSON string of the whole block
    StructField("changed_movies", StringType(), True),   # <-- raw JSON string of the whole block
])

In [0]:
# Create Audit Execution Log Table
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.tmdb.pipeline_execution_logs (
    log_id STRING,
    layer_processed STRING,
    load_type STRING,
    batch_id STRING,
    execution_start_time TIMESTAMP,
    execution_end_time TIMESTAMP,
    status STRING,
    records_processed LONG,
    error_message STRING
)
USING DELTA
""")

def log_pipeline_execution(layer, load_type, batch_id, start_time, end_time, status, records, error_msg="NONE"):
    log_data = [(
        str(uuid.uuid4()),
        layer,
        load_type,
        batch_id,
        start_time,
        end_time,
        status,
        records,
        error_msg
    )]
    schema = spark.table("workspace.tmdb.pipeline_execution_logs").schema
    log_df = spark.createDataFrame(log_data, schema=schema)
    log_df.write.format("delta").mode("append").saveAsTable("workspace.tmdb.pipeline_execution_logs")
    print(f"Logged execution for layer '{layer}': Status={status}, Records={records}")

In [0]:
from pyspark.sql import functions as F

full_load_path     = "/Volumes/workspace/tmdb/raw/full_load/full_load.json"
incremental_path   = "/Volumes/workspace/tmdb/raw/incremental/incremental_load.json"

# ---------- BRONZE: FULL LOAD ----------
df_full_raw = (
    spark.read
    .schema(full_load_envelope_schema)          # explicit, NO inference
    .option("multiLine", "true")
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_rescued_data")
    .json(full_load_path)
)

df_full_exploded = (
    df_full_raw
    .select(
        F.col("pulled_at"),
        F.col("source_endpoint"),
        F.col("year_range"),
        F.explode(F.col("results")).alias("raw_record")
    )
    .withColumn("source_collection", F.lit("full_load"))
)

# ---------- BRONZE: INCREMENTAL ----------
df_inc_raw = (
    spark.read
    .schema(incremental_envelope_schema)
    .option("multiLine", "true")
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_rescued_data")
    .json(incremental_path)
)

# Trending block — trending_today is a raw JSON string; parse it shallowly to explode its results
trending_parsed = (
    df_inc_raw
    .select(F.col("pulled_at"), F.col("trending_today"))
    .withColumn("tt", F.from_json("trending_today",
        StructType([
            StructField("source_endpoint", StringType(), True),
            StructField("record_count",    LongType(),   True),
            StructField("results",         ArrayType(StringType()), True),
        ])
    ))
    .select(
        F.col("pulled_at"),
        F.col("tt.source_endpoint").alias("source_endpoint"),
        F.explode(F.col("tt.results")).alias("raw_record")
    )
    .withColumn("source_collection", F.lit("trending_today"))
)

# Changed_movies block — same idea
changed_parsed = (
    df_inc_raw
    .select(F.col("pulled_at"), F.col("changed_movies"))
    .withColumn("cm", F.from_json("changed_movies",
        StructType([
            StructField("source_endpoint",       StringType(), True),
            StructField("changed_id_count",      LongType(),   True),
            StructField("resolved_record_count", LongType(),   True),
            StructField("results",               ArrayType(StringType()), True),
        ])
    ))
    .select(
        F.col("pulled_at"),
        F.col("cm.source_endpoint").alias("source_endpoint"),
        F.explode(F.col("cm.results")).alias("raw_record")
    )
    .withColumn("source_collection", F.lit("changed_movies"))
)

# ---------- UNION + WRITE BRONZE ----------
common_cols = ["pulled_at", "source_endpoint", "source_collection", "raw_record"]

df_bronze = (
    df_full_exploded.select(*common_cols)
    .unionByName(trending_parsed.select(*common_cols))
    .unionByName(changed_parsed.select(*common_cols))
    .withColumn("load_timestamp", F.current_timestamp())
    .withColumn("batch_id", F.lit(batch_id))
)

(
    df_bronze.write
    .format("delta")
    .mode("overwrite")                    # Bronze is a landing zone for THIS batch
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.tmdb.bronze_movies")
)

print(f"Bronze written: {spark.table('workspace.tmdb.bronze_movies').count()} rows")

In [0]:
#task 4 a
df_b = spark.table("workspace.tmdb.bronze_movies")

print(f"Total rows: {df_b.count()}")
print("\n--- Breakdown by source_collection ---")
df_b.groupBy("source_collection").count().orderBy("source_collection").show()

print("\n--- Bronze schema ---")
df_b.printSchema()

print("\n--- Sample raw_record (full_load) ---")
display(df_b.filter("source_collection = 'full_load'").select("raw_record").limit(1))

print("\n--- Sample raw_record (trending_today) ---")
display(df_b.filter("source_collection = 'trending_today'").select("raw_record").limit(1))

print("\n--- Sample raw_record (changed_movies) ---")
display(df_b.filter("source_collection = 'changed_movies'").select("raw_record").limit(1))

print("\n--- Distinct batch_id + load_timestamp present? ---")
df_b.select("batch_id", "load_timestamp").distinct().show(5, truncate=False)